# Chatbot for Internship Support
**Task 6 NLP Chatbot (Retrieval-Based)**

Pipeline: Sentence-BERT embeddings (MiniLM) + cosine similarity retrieval over a FAQ + support ticket dataset. Returns the verified answer directly no generative rephrasing layer (see Pipeline Summary at the end for why).

In [1]:
# Install the sentence-transformers library
# This gives us pretrained Sentence-BERT models for generating text embeddings
!pip install sentence-transformers -q

In [2]:
# Import all the libraries needed for this project
import pandas as pd
import numpy as np
import pickle
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

##  Upload and Load the Dataset
Upload `answers.csv`, `questions.csv`, and `support_tickets.csv` here.

In [4]:
# Upload the 3 CSV files from local system
from google.colab import files
uploaded = files.upload()   # select answers.csv, questions.csv, support_tickets.csv

Saving questions.csv to questions.csv
Saving support_tickets.csv to support_tickets.csv


In [7]:
# Load the datasets into pandas dataframes
answers_df = pd.read_csv("answers.csv")
questions_df = pd.read_csv("questions.csv")
tickets_df = pd.read_csv("support_tickets.csv")

# Quick check that everything loaded correctly
print("Answers:", answers_df.shape)
print("Questions:", questions_df.shape)
print("Tickets:", tickets_df.shape)
answers_df.head()

Answers: (20, 3)
Questions: (88, 3)
Tickets: (36, 3)


,answer_id,category,answer
0,A001,Task submission,"To submit your task, log in to your dashboard,..."
1,A002,Deadlines,Each task's deadline is mentioned on its task ...
2,A003,Mentor contact,You can reach your mentor through the official...
3,A004,Certificates,Certificates are issued after you successfully...
4,A005,Attendance,Attendance is tracked based on your task submi...


## Exploratory Data Analysis (EDA)
Before building anything, checking the dataset for missing values, duplicates and how the data is distributed across categories.

In [8]:
# Check for missing values in each file
print("Missing values:")
print("answers.csv:\n", answers_df.isnull().sum())
print("\nquestions.csv:\n", questions_df.isnull().sum())
print("\nsupport_tickets.csv:\n", tickets_df.isnull().sum())

Missing values:
answers.csv:
 answer_id    0
category     0
answer       0
dtype: int64

questions.csv:
 question     0
answer_id    0
category     0
dtype: int64

support_tickets.csv:
 ticket_message    0
answer_id         0
category          0
dtype: int64


In [9]:
# Check for exact duplicate rows
print("Duplicate rows in questions.csv:", questions_df.duplicated().sum())
print("Duplicate rows in support_tickets.csv:", tickets_df.duplicated().sum())

# Check that every answer_id used in questions/tickets actually exists in answers.csv
valid_ids = set(answers_df["answer_id"])
missing_in_questions = set(questions_df["answer_id"]) - valid_ids
missing_in_tickets = set(tickets_df["answer_id"]) - valid_ids
print("answer_ids in questions.csv with no matching answer:", missing_in_questions)
print("answer_ids in support_tickets.csv with no matching answer:", missing_in_tickets)

Duplicate rows in questions.csv: 0
Duplicate rows in support_tickets.csv: 0
answer_ids in questions.csv with no matching answer: set()
answer_ids in support_tickets.csv with no matching answer: set()


In [10]:
# How many questions/tickets exist per category checking for class imbalance
print("Questions per category:")
print(questions_df["category"].value_counts())

print("\nTickets per category:")
print(tickets_df["category"].value_counts())

Questions per category:
category
Task submission         6
Deadlines               5
Mentor contact          5
Certificates            5
Profile update          5
Dashboard issues        5
Login problems          5
Attendance              4
Password reset          4
Internship duration     4
Task evaluation         4
Resubmission            4
Feedback                4
Completion status       4
Technical issues        4
Team projects           4
GitHub submission       4
Streamlit deployment    4
Project requirements    4
AI usage policy         4
Name: count, dtype: int64

Tickets per category:
category
Task submission         2
Deadlines               2
Mentor contact          2
Certificates            2
Attendance              2
Profile update          2
Dashboard issues        2
Login problems          2
Password reset          2
Internship duration     2
Task evaluation         2
Resubmission            2
Feedback                2
Completion status       2
Technical issues        2

In [11]:
# Look at the length word count of questions vs tickets
# This tells us if tickets are genuinely more "informal/short" compared to clean FAQ questions
questions_df["word_count"] = questions_df["question"].apply(lambda x: len(x.split()))
tickets_df["word_count"] = tickets_df["ticket_message"].apply(lambda x: len(x.split()))

print("Average words per FAQ question:", round(questions_df["word_count"].mean(), 2))
print("Average words per support ticket:", round(tickets_df["word_count"].mean(), 2))

Average words per FAQ question: 7.77
Average words per support ticket: 11.11


## Light Preprocessing
For a Sentence-BERT model, heavy preprocessing removing stopwords, stemming, lowercasing everything is intentionally **avoided** here because MiniLM was pretrained on natural full sentences stripping words out would actually reduce embedding quality instead of improving it. So only basic cleaning is done: trimming extra whitespace and removing duplicate rows.

In [12]:
# Strip leading/trailing whitespace from all text columns
questions_df["question"] = questions_df["question"].str.strip()
tickets_df["ticket_message"] = tickets_df["ticket_message"].str.strip()
answers_df["answer"] = answers_df["answer"].str.strip()

# Drop any exact duplicate rows found during EDA
questions_df = questions_df.drop_duplicates(subset=["question"]).reset_index(drop=True)
tickets_df = tickets_df.drop_duplicates(subset=["ticket_message"]).reset_index(drop=True)

# Drop any rows where the question/ticket ended up empty after stripping
questions_df = questions_df[questions_df["question"].str.len() > 0].reset_index(drop=True)
tickets_df = tickets_df[tickets_df["ticket_message"].str.len() > 0].reset_index(drop=True)

print("Questions after cleaning:", questions_df.shape)
print("Tickets after cleaning:", tickets_df.shape)

Questions after cleaning: (88, 4)
Tickets after cleaning: (36, 4)


## Combine Questions + Tickets into One Query Pool
Both FAQ questions and support tickets point to the same `answer_id`, so I am merging them into a single dataframe that the model will search over. This way, the chatbot can match either a clean FAQ-style question or a real informal user message.

In [13]:
# Rename ticket_message column to 'question' so both dataframes have the same structure
tickets_renamed = tickets_df.rename(columns={"ticket_message": "question"})

# Combine questions.csv and support_tickets.csv into one dataframe
all_queries_df = pd.concat(
    [questions_df[["question", "answer_id", "category"]],
     tickets_renamed[["question", "answer_id", "category"]]],
    ignore_index=True
)

print("Total queries in the pool:", len(all_queries_df))
all_queries_df.sample(5)

Total queries in the pool: 124


,question,answer_id,category
43,"The reset password link isn't working, what now?",A009,Password reset
115,The progress bar on my dashboard isn't showing...,A014,Completion status
54,"I need to redo my task, how does resubmission ...",A012,Resubmission
82,Is there a detailed instructions page for tasks?,A019,Project requirements
67,"My file upload keeps failing, any fix?",A015,Technical issues


## Load the Sentence-BERT Model
Using `all-MiniLM-L6-v2`  a small, fast Sentence-BERT model that converts a sentence into a 384-dimension embedding vector. Sentences with similar meaning end up with vectors that are close to each other which is exactly what we need for FAQ matching.

In [14]:
# Load the pretrained MiniLM model
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Model loaded. Embedding size:", model.get_sentence_embedding_dimension())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded. Embedding size: 384


/tmp/ipykernel_2669/1434551666.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Model loaded. Embedding size:", model.get_sentence_embedding_dimension())


##  Generate Embeddings for Every Question in the Pool
Every question and ticket in `all_queries_df` gets converted into an embedding vector. These embeddings are what I will compare a new user query against.

In [15]:
# Encode all questions/tickets into embeddings
query_embeddings = model.encode(
    all_queries_df["question"].tolist(),
    show_progress_bar=True
)

print("Embeddings shape:", query_embeddings.shape)   # (num_queries, 384)

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Embeddings shape: (124, 384)


##  Build the Retrieval Function
This is the core chatbot logic:
1. Take the user's message and encode it using the same MiniLM model.
2. Compute cosine similarity between the user's embedding and every embedding in the pool.
3. Pick the question with the highest similarity score.
4. Look up the matching `answer_id` in `answers.csv` and return that answer.
5. If the best similarity score is below a threshold, return a fallback message instead of a wrong answer.

In [16]:
# Threshold below which we don't trust the match and return a fallback response
SIMILARITY_THRESHOLD = 0.45

# Build a quick lookup dictionary: answer_id -> answer text
answer_lookup = dict(zip(answers_df["answer_id"], answers_df["answer"]))

def get_chatbot_response(user_query, top_k=1):
    # Encode the user's query with the same model used for the dataset
    query_embedding = model.encode([user_query])

    # Compare it against every embedding in the pool
    similarities = cosine_similarity(query_embedding, query_embeddings)[0]

    # Find the index of the closest match
    best_idx = np.argmax(similarities)
    best_score = similarities[best_idx]

    # If confidence is too low, don't guess ask the user to contact support instead
    if best_score < SIMILARITY_THRESHOLD:
        return {
            "answer": "I'm not fully sure about that. Please reach out to the support team for help with this.",
            "matched_question": None,
            "confidence": float(best_score)
        }

    matched_row = all_queries_df.iloc[best_idx]
    answer_id = matched_row["answer_id"]

    return {
        "answer": answer_lookup[answer_id],
        "matched_question": matched_row["question"],
        "category": matched_row["category"],
        "confidence": float(best_score)
    }

## Test the Chatbot
Trying a few sample queries, including ones worded differently from the training data, to check that semantic matching is actually working not just exact keyword matching.

In [17]:
# Test queries some are close to training data some are worded very differently
test_queries = [
    "how do i turn in my project",
    "i cant login into my account",
    "when do i get my certificate",
    "my app is not deploying properly",
    "what's the weather today"   # unrelated query, should trigger fallback
]

for q in test_queries:
    result = get_chatbot_response(q)
    print("User:", q)
    print("Bot :", result["answer"])
    print("Confidence:", round(result["confidence"], 3))
    print("-" * 60)

User: how do i turn in my project
Bot : To submit your task, log in to your dashboard, open the current task card, and click the 'Submit Work' button. Make sure all required files are attached before submitting.
Confidence: 0.748
------------------------------------------------------------
User: i cant login into my account
Bot : If you're unable to log in, double-check your registered email and password. If the issue persists, reset your password or contact support.
Confidence: 0.886
------------------------------------------------------------
User: when do i get my certificate
Bot : Certificates are issued after you successfully complete and submit all required tasks. It usually takes a few working days to generate once your final task is approved.
Confidence: 0.956
------------------------------------------------------------
User: my app is not deploying properly
Bot : If deployment is required, deploy your Streamlit app (e.g. via Streamlit Community Cloud) and share the live app li

## Save the Artifacts for Deployment
Saving everything the Streamlit app will need the embeddings, the query dataframe, the answers lookup, and the model name into a single `.pkl` file. This follows the same workflow as my previous tasks: train in Colab, save `.pkl`, then load it in the VS Code Streamlit app.

**Note:** I tried adding a Flan-T5 layer on top of retrieval to rephrase answers in a friendlier tone, but both `flan-t5-small` and `flan-t5-base` ended up either over-summarizing the answers or adding details that weren't in the original dataset (hallucinating). Since accuracy matters more than tone here, I removed that layer and the chatbot returns the verified answer from `answers.csv` directly.

In [18]:
# Package everything needed for inference into one dictionary
chatbot_artifacts = {
    "model_name": "all-MiniLM-L6-v2",
    "query_embeddings": query_embeddings,
    "all_queries_df": all_queries_df,
    "answer_lookup": answer_lookup,
    "similarity_threshold": SIMILARITY_THRESHOLD
}

# Save to a pickle file
with open("chatbot_artifacts.pkl", "wb") as f:
    pickle.dump(chatbot_artifacts, f)

print("Saved chatbot_artifacts.pkl")

Saved chatbot_artifacts.pkl


In [19]:
# Download the .pkl file to local machine, to move into the VS Code project folder
from google.colab import files
files.download("chatbot_artifacts.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Pipeline Summary

**Type:** Retrieval-based FAQ chatbot. Every response comes directly from the verified `answers.csv`, matched to the user's message using semantic similarity.

**Steps:**
1. **Data:** `answers.csv` unique, verified answers + `questions.csv`/`support_tickets.csv` many phrasings pointing to each answer.
2. **EDA and light preprocessing:** checked for missing values, duplicates, category balance, and question length; cleaning kept minimal whitespace trimming, de-duplication only since Sentence-BERT performs best on natural, unmodified sentences.
3. **Embedding model:** `all-MiniLM-L6-v2` (Sentence-BERT) turns every sentence into a 384-dimension vector that captures its meaning.
4. **Matching:** for a new user message, the same model creates its embedding, then **cosine similarity** compares it against every stored question/ticket embedding.
5. **Answer lookup:** the closest match's `answer_id` is used to fetch the exact, verified answer text from `answers.csv` returned as it is.
6. **Fallback:** if the best similarity score is below `0.45` the bot admits it isn't sure instead of returning a wrong answer.
7. **Deployment prep:** embeddings + lookup tables are saved as one `.pkl` file, loaded directly in the Streamlit app.

**Why no generative rephrasing layer:** I tested adding Flan-T5 (both `small` and `base`) on top of retrieval to reword answers in a friendlier tone. In practice it either over-summarized the answers dropping real details or hallucinated steps that don't exist in the actual dataset (e.g. inventing a 'Complete Tasks' button). For a support chatbot, factual accuracy matters more than tone, so I removed the generative layer and kept the pipeline purely retrieval-based this still fully satisfies the "automate real-time responses" requirement, since retrieval returns an answer instantly, without any human needing to intervene.